# Multi-Task Contextual Decision-Making (Yang19 Cognitive Tasks)

This environment implements a suite of 20 context-dependent cognitive tasks inspired by the work of [Yang et al. (2019)](https://pubmed.ncbi.nlm.nih.gov/30643294/#:~:text=The%20brain%20has%20the%20ability,task%20representations%2C%20a%20critical%20feature). In these tasks, the agent must flexibly switch between different context rules on each trial, similar in spirit to the classic [context-dependent decision-making](https://neurogym.github.io/neurogym/latest/api/envs/#neurogym.envs.native.contextdecisionmaking.ContextDecisionMaking) experiment of [Mante et al. (2013)](https://www.nature.com/articles/nature12742). The key features of this multi-task environment are:

1. **Two-Choice Outputs** (`dim_ring=2`): All tasks share a common action space represented as positions on a ring. With `dim_ring = 2`, there are two choice outputs located at 0° and 180° on this ring (interpreted as "left" and "right" choices, respectively). The agent's goal on each trial is to select the correct one of these two outputs.
2. **Stimulus Inputs with Modality-Specific Evidence**: Each task provides sensory evidence (stimuli) that may come from one or two modalities. The stimuli are often encoded via a cosine-tuned input bump centered on one of the choice positions, with its amplitude representing the evidence strength. For example, a stimulus favoring the left choice might produce an activity peak at 0° on the input ring.
3. **Randomized Ground-Truth Choices**: The correct choice (ground truth) is randomized on each trial (for two choices: left or right). This means on some trials left is the correct answer, and on others right is correct, preventing the agent from biasing toward one action.
4. **Variable Difficulty Evidence**: The strength of the stimulus evidence (e.g., coherence level in a motion stimulus) is also randomly sampled each trial. Sometimes the evidence strongly favors one choice, and other times it is weak or ambiguous.

This collection of 20 tasks exercises various cognitive skills, including working memory (e.g., remembering stimuli across a delay in matching tasks), perceptual decision-making (integrating noisy sensory evidence over time), rule-based categorization, and inhibitory control (suppressing or initiating actions under certain rules). All tasks are implemented in a consistent format so that a single neural network agent can learn them together via supervised training.

In this notebook, we will:

1. Train an agent on the 20-task suite using supervised learning. We will generate trial data from the environment and train a recurrent neural network to predict the correct choices.
2. Evaluate the agent's performance and behavior across tasks. After training, we will examine how well the agent learned each of the 20 tasks and whether it can flexibly switch contexts from trial to trial.


# 0. Install Dependencies

To begin, install the `neurogym` package. This will automatically install all required dependencies, including Stable-Baselines3.

For detailed instructions on how to install `neurogym` within a conda environment or in editable mode, refer to the [installation instructions](https://github.com/neurogym/neurogym?tab=readme-ov-file#installation).


# 1. Training an Agent on Yang19 Cognitive Tasks


## 1.1 Environment Setup and Initial Agent Behavior

Let's begin by creating and exploring the environment using the `yang19` collection of tasks from NeuroGym. We’ll use the default configuration for all parameters except `dim_ring`, which we set to 2 in order to represent two alternative choices (left/right) arranged on a ring.

To get a sense of the environment dynamics, we’ll visualize a couple of representative tasks (`dm1` and `multidm`) to better understand their structure and how the agent is expected to interact with them.


### 1.1.1 Import Libraries


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch

import neurogym as ngym
from neurogym.wrappers import ScheduleEnvs, Monitor
from neurogym.utils import plot_env, RandomSchedule, SequentialSchedule
from neurogym.envs import get_collection
from torch.nn import functional as F
from IPython.display import clear_output
warnings.filterwarnings("ignore")
clear_output()

### 1.1.2 Environment Setup


In [ ]:
# Environment parameters
# This setting is low to speed up testing, we recommend setting it to at least 1000
EVAL_TRIALS = 100
dt = 100
dim_ring = 2  # Number of choices in the ring representation

rewards = {
    "abort": -0.1,
    "correct": +1.0,
    "fail": 0.0
}
sigma = 1.0 # Standard deviation of the Gaussian noise in the ring representation

We wrap the `yang19` environments using NeuroGym’s `ScheduleEnvs` wrapper, which allows us to interleave multiple tasks into a single training loop. Each cognitive task is instantiated as a separate environment, and a scheduling policy (`RandomSchedule`) determines which task is sampled on each trial. When `env_input=True`, a one-hot vector is appended to the observation to indicate the currently active task, allowing the agent to learn task-specific behavior within a unified architecture.


In [ ]:
kwargs = {'dt': dt, 'dim_ring': dim_ring}
seq_len = 100

# Make supervised dataset
tasks = get_collection('yang19')
envs = [ngym.make(task, **kwargs) for task in tasks]
schedule = RandomSchedule(len(envs))
# schedule = SequentialSchedule(len(envs))
env = ScheduleEnvs(envs, schedule=schedule, env_input=True)

# Print environment specifications
print("Trial timing (in milliseconds):")
print(env.timing)

print("\nObservation space structure:")
print(env.observation_space)

print("\nAction space structure:")
print(env.action_space)
print("Action mapping:")
print(env.action_space.name)

### 1.1.3 Random Agent Behavior

Let's now plot the behavior of a random agent on the task. The agent will randomly choose between the two options (left/right, blue line), and we will visualize its behavior over 5 trials. We will also plot the reward received by the agent at each time step, as well as the performance on each trial. Note that performance is only defined at the end of a trial: it is 1 if the agent made the correct choice, and 0 otherwise.


In [ ]:
obs, info = env.reset()

eval_monitor = Monitor(env)
print("\nEvaluating random policy performance...")
metrics = eval_monitor.evaluate_policy(num_trials=EVAL_TRIALS)
print(f"\nRandom policy metrics ({EVAL_TRIALS:,} trials):")
print(f"Mean performance: {metrics['mean_performance']:.4f}")
print(f"Mean reward: {metrics['mean_reward']:.4f}")

# 2. Learning the Task as a Supervised Problem

We will now train the agent using supervised learning. NeuroGym provides functionality to generate a dataset directly from the environment, allowing us to sample batches of inputs and corresponding labels for training.


## 2.1 Converting the Environment to a Supervised Dataset


### 2.1.1 Dataset Setup


In [ ]:
seq_len = 100
print(f"Using sequence length: {seq_len}")

# Make supervised dataset
batch_size = 64
print(f"Creating dataset with batch_size={batch_size}")
dataset = ngym.Dataset(env, batch_size=batch_size, seq_len=seq_len)

env = dataset.env

# Extract dimensions
ob_size = env.observation_space.shape[0]
act_size = env.action_space.n
# 20 observations for one-hot encoding of the 20 environments, 1 for fixation,
# 2 for each modality (2 modalities, 4 observations total), for a total of 25 observations
print(f"Observation size: {ob_size}")
print(f"Action size: {act_size}")

# Get a batch of data
# inputs, target = dataset()
# print(f"Input batch shape: {inputs.shape}")
# print(f"Target batch shape: {target.shape}")
# print(inputs[90])

# Set device
device = 'cuda:0'
print(f"Using device: {device}")

## 2.2 Training and Evaluating a Neural Network Model


### 2.2.1 Training the Model


In [ ]:
# from pdm_model import Net, SimpleNet, DANN, ColEI
from net.colei import ColEI
from net.dann import DANN
from net.simple import SimpleNet
from net.eisep import EISepNet
from net.dbp import DBPN
from net.common import init_model

# This setting is low to speed up testing, we recommend setting it to at least 1000
def train(net, epochs=2000, lyapunov_every=100, lr=1e-3, wd=1e-5, batch_size=64):
    seq_len = 100
    dataset = ngym.Dataset(env, batch_size=batch_size, seq_len=seq_len)

    net.train()
    # This weighting deprioritizes class 0 while keeping the other 16 classes equally important,
    # aligning with the reward distribution idea from the RL setting
    class_weights = torch.tensor([0.05] + [1.0]*(act_size - 1)).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # Training loop
    loss_history = []
    loss_log = np.zeros((epochs, 3))
    gradient_log = np.zeros((epochs, 1))
    spectral_radius_log = np.zeros((epochs, 1))
    rec_balance_log = np.zeros((epochs, seq_len))  # Track input balance
    n_lyapunov = epochs // lyapunov_every
    lyapunov_spectrum_log = np.zeros((n_lyapunov, net.hidden_size))  # Track Lyapunov spectrum
    np.set_printoptions(threshold=np.inf)

    for i in range(epochs):
        # Get a batch of data
        inputs, targets = dataset()
        # print(inputs.shape, targets.shape)
        # print(inputs[..., -20:].argmax(axis=-1))
        # Convert to PyTorch tensors
        inputs = torch.from_numpy(inputs).float().to(device)
        targets = torch.from_numpy(targets).long().to(device)

        # Initialize hidden state
        # hidden = sl_model.init_hidden(inputs.size(1), device)


        # Forward pass with hidden state tracking
        outputs, eimask_logit, hidden = net(inputs)

        # Reshape for CrossEntropyLoss
        outputs_flat = outputs.reshape(-1, outputs.size(2))
        targets_flat = targets.reshape(-1)

        # Calculate loss
        # Weight the loss to account for class imbalance (very low weight to 0s, higher weights to 1s and 2s)
        decoding_loss = criterion(outputs_flat, targets_flat)


        if net.entropy_loss:
            # eimask_prob = F.softmax(eimask_logit, dim=-1)
            # eimask_entropy = -torch.sum(eimask_prob * torch.log(eimask_prob + 1e-10), dim=-1)
            # entropy_loss = torch.mean(eimask_entropy)  # Add entropy regularization
            entropy_loss = net.get_ei_entropy_loss()
            loss_log[i, 2] = entropy_loss.item()
        else:
            entropy_loss = 0.0


        if isinstance(net, SimpleNet):
            l1_loss = net.lstm.Wrec_free.abs().mean()
        else:
            l1_loss = 0

        loss = 1 * entropy_loss + decoding_loss + l1_loss * 0

        # Backward pass and optimize
        loss.backward()
        # gradient_log[i, 0] = net.lstm.Wrec_free.grad.norm(2).item()
        if isinstance(net, DANN):
            gradient_log[i, 0] = net.danncell.Wex.grad.norm(2).item()
            net.update(lr=lr, max_grad_norm=None)
            net.zero_grad()
        elif isinstance(net, ColEI):
            gradient_log[i, 0] = net.colcell.W_pos.grad.norm(2).item()
            net.update(lr=lr, max_grad_norm=10)
            net.zero_grad()
        elif isinstance(net, DBPN):
            gradient_log[i, 0] = net.rnn.weight_hh_l0.grad.norm(2).item()
            net.update(optimizer)
            optimizer.zero_grad()
        else:
            gradient_log[i, 0] = net.lstm.Wrec_free.grad.norm(2).item()
            optimizer.step()
            optimizer.zero_grad()

        # Log spectral radius
        weight = net.get_actual_weight()
        try:
            spectral_radius = torch.linalg.eigvals(weight).abs().max().item()
        except:
            spectral_radius = torch.nan
        spectral_radius_log[i, 0] = spectral_radius

        rec_balance_log[i] = hidden[-1].detach().cpu().numpy()

        # print statistics
        loss_history.append(loss.item())
        loss_log[i, 0] = loss.item()
        loss_log[i, 1] = decoding_loss.item()
        if i % lyapunov_every == 0:
            print('Epoch [{}/{}], Loss: {:.4f}'.format(i, epochs, loss.item()))
        #     # Calculate Lyapunov spectrum
            # lyapunov_spectrum, _ = net.compute_lyapunov_spectrum(inputs)
            # lyapunov_spectrum_log[i // lyapunov_every] = lyapunov_spectrum.detach().cpu().numpy()


    print('Finished Training')
    return net, loss_log, gradient_log, spectral_radius_log, rec_balance_log, lyapunov_spectrum_log
    # Plot the loss curve
    # plt.figure(figsize=(8, 4))
    # plt.plot(loss_history)
    # plt.title('Training Loss')
    # plt.xlabel('Iterations')
    # plt.ylabel('Loss (50-iteration moving average)')
    # plt.grid(True, alpha=0.3)
    # plt.savefig('./output/multitask/yang19_loss_curve.png')
    # plt.show()

### 2.2.2 Evaluate the Model's Performance


In [ ]:
def test(net):
    verbose = False
    # Evaluate performance of the trained model
    net.eval()
    # Initialize storage for each environment
    sl_mean_performance = {}
    for e_i in env.envs:
        sl_mean_performance[e_i.spec.id] = []

    # Evaluate for specified number of trials
    print(f"Evaluating model performance across {EVAL_TRIALS} trials...\n")

    for env_idx, e_i in enumerate(env.envs):
        data = {"action": [], "gt": [], "trial": []}
        total_correct = 0
        for trial_idx in range(EVAL_TRIALS):
            trial = e_i.new_trial()
            data["trial"].append(trial)
            ob, gt = e_i.ob, e_i.gt
            data["gt"].append(gt[-1])
            trial_length = ob.shape[0]

            # Add one-hot encoding for the environment
            env_one_hot = np.zeros((trial_length, len(env.envs)))
            env_one_hot[:, env_idx] = 1.0  # Set the current environment index to 1

            # Concatenate original observation with one-hot encoding
            ob_with_env = np.concatenate([ob, env_one_hot], axis=1)

            ob_with_env = ob_with_env[:, np.newaxis, :]  # Add batch dimension

            inputs = torch.from_numpy(ob_with_env).float().to(device)
            # hidden = sl_model.init_hidden(1, device)

            with torch.no_grad():
                outputs, _, hidden = net(inputs)
                pred_actions = torch.argmax(outputs, dim=2)
                data["action"].append(pred_actions[-1, 0].cpu().numpy())

            decision_idx = trial_length - 1
            is_correct = (gt[decision_idx] == pred_actions[decision_idx, 0].cpu().numpy())
            total_correct += is_correct

        accuracy = total_correct / EVAL_TRIALS
        sl_mean_performance[e_i.spec.id].append(accuracy)
        for key in data:
            if key != "trial":
                data[key] = np.array(data[key])


    # Print average performance
    print("Average performance across all environments:")
    mean_acc_all = np.array([np.mean(sl_mean_performance[e_i.spec.id]) for e_i in env.envs])
    for e_i in env.envs:
        mean_acc = np.mean(sl_mean_performance[e_i.spec.id])
        print(f"{e_i.spec.id}: {mean_acc:.4f}")
    return mean_acc_all





def plot_lyapunov_spectrum(lyapunov_spectrum_log_ei, lyapunov_spectrum_log_simple):
    n_lyapunov = lyapunov_spectrum_log_ei.shape[1]
    hidden_size = lyapunov_spectrum_log_ei.shape[2]
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    fontsize = 15
    mean_ei = np.mean(lyapunov_spectrum_log_ei, axis=0)
    std_ei = np.std(lyapunov_spectrum_log_ei, axis=0)  # population std; use ddof=1 for sample std
    mean_simple = np.mean(lyapunov_spectrum_log_simple, axis=0)
    std_simple = np.std(lyapunov_spectrum_log_simple, axis=0)
    argsort_ei = np.flip(np.argsort(mean_ei, axis=1), axis=1)
    argsort_simple = np.flip(np.argsort(mean_simple, axis=1), axis=1)
    sorted_mean_ei = np.take_along_axis(mean_ei, argsort_ei, axis=1)
    sorted_std_ei = np.take_along_axis(std_ei, argsort_ei, axis=1)
    sorted_mean_simple = np.take_along_axis(mean_simple, argsort_simple, axis=1)
    sorted_std_simple = np.take_along_axis(std_simple, argsort_simple, axis=1)
    alpha_list = np.linspace(0.1, 1.0, n_lyapunov)
    for i in range(n_lyapunov):
        ax.plot(sorted_mean_ei[i], label=f'Lyapunov {i+1}', alpha=alpha_list[i], color='C0')
        ax.plot(sorted_mean_simple[i], label=f'Lyapunov {i+1}', alpha=alpha_list[i], color='C1')
        # ax[0].fill_between(np.arange(hidden_size), sorted_mean_ei[i] - sorted_std_ei[i], sorted_mean_ei[i] + sorted_std_ei[i], alpha=0.2, color='C0')
        # ax[1].plot(sorted_mean_simple[i], label=f'Lyapunov {i+1}', alpha=alpha_list[i], color='C1')
        # ax[1].fill_between(np.arange(hidden_size), sorted_mean_simple[i] - sorted_std_simple[i], sorted_mean_simple[i] + sorted_std_simple[i], alpha=0.2, color='C1')
    # ax[0].set_title('Lyapunov spectrum (EI)', fontsize=fontsize)
    # ax[1].set_title('Lyapunov spectrum (simple)', fontsize=fontsize)
    ax.set_xlabel('Lyapunov index', fontsize=fontsize)
    # ax[1].set_xlabel('Lyapunov index', fontsize=fontsize)
    ax.set_ylabel('Lyapunov exponent', fontsize=fontsize)
    # ax[1].set_ylabel('Lyapunov exponent', fontsize=fontsize)
    # ax[0].legend(fontsize=fontsize)
    # ax[1].legend(fontsize=fontsize)
    # for a in ax:
    #     a.tick_params(labelsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    plt.tight_layout()
    plt.savefig('./output/multitask/yang19_lyapunov_spectrum.png')
    plt.show()

def plot_lyapunov_spectrum_separately(lyapunov_spectrum_log_ei, lyapunov_spectrum_log_simple, lyapunov_every):
    n_lyapunov = lyapunov_spectrum_log_ei.shape[1]
    hidden_size = lyapunov_spectrum_log_ei.shape[2]
    fig, ax = plt.subplots(4, 5, figsize=(16, 10), sharex=True)
    ax = ax.flatten()
    fontsize = 15
    mean_ei = np.mean(lyapunov_spectrum_log_ei, axis=0)
    std_ei = np.std(lyapunov_spectrum_log_ei, axis=0)  # population std; use ddof=1 for sample std
    mean_simple = np.mean(lyapunov_spectrum_log_simple, axis=0)
    std_simple = np.std(lyapunov_spectrum_log_simple, axis=0)
    argsort_ei = np.flip(np.argsort(mean_ei, axis=1), axis=1)
    argsort_simple = np.flip(np.argsort(mean_simple, axis=1), axis=1)
    sorted_mean_ei = np.take_along_axis(mean_ei, argsort_ei, axis=1)
    sorted_std_ei = np.take_along_axis(std_ei, argsort_ei, axis=1)
    sorted_mean_simple = np.take_along_axis(mean_simple, argsort_simple, axis=1)
    sorted_std_simple = np.take_along_axis(std_simple, argsort_simple, axis=1)
    # alpha_list = np.linspace(0.1, 1.0, n_lyapunov)
    for i in range(n_lyapunov):
        ax[i].plot(sorted_mean_ei[i], label='EI', color='C0')
        ax[i].fill_between(np.arange(hidden_size), sorted_mean_ei[i] - sorted_std_ei[i], sorted_mean_ei[i] + sorted_std_ei[i], alpha=0.2, color='C0')
        ax[i].plot(sorted_mean_simple[i], label='simple', color='C1')
        ax[i].fill_between(np.arange(hidden_size), sorted_mean_simple[i] - sorted_std_simple[i], sorted_mean_simple[i] + sorted_std_simple[i], alpha=0.2, color='C1')
        ax[i].set_title(f'Epoch {i+1}*{lyapunov_every}', fontsize=fontsize)
        if i > 14:
            ax[i].set_xlabel('Lyapunov index', fontsize=fontsize)
        if i % 5 == 0:
            ax[i].set_ylabel('Lyapunov exponent', fontsize=fontsize)
        ax[i].legend(fontsize=fontsize)
        ax[i].tick_params(labelsize=fontsize)
    plt.tight_layout()
    plt.savefig('./output/multitask/yang19_lyapunov_spectrum_separately.png')
    plt.show()


In [ ]:
import os, pickle
n_seed = 5
epochs = 3000
hidden_size = 64
def multiple_train_multilr(model_name, lrs):

    lyapunov_every = epochs//20
    hidden_size = 64
    n_lr = len(lrs)
    wd_ei = 0
    batch_size = 32
    test_EI_performance = np.zeros((n_lr, n_seed, len(env.envs)))
    loss_log_ei = np.zeros((n_lr, n_seed, epochs, 3))
    gradient_log_ei = np.zeros((n_lr, n_seed, epochs, 1))
    spectral_radius_log_ei = np.zeros((n_lr, n_seed, epochs, 1))
    rec_balance_log_ei = np.zeros((n_lr, n_seed, epochs, seq_len))
    lyapunov_spectrum_log_ei = np.zeros((n_lr, n_seed, epochs // lyapunov_every, hidden_size))  # Track Lyapunov spectrum for EI
    net_state_dicts = [[None]*n_seed for _ in range(n_lr)]

    for j, lr in enumerate(lrs):
        print(f"Learning rate {j+1}/{len(lrs)}: {lr}")
        for seed in range(n_seed):
            print(f"Seed {seed + 1}/{n_seed}")
            torch.manual_seed(seed)
            np.random.seed(seed)
            print('device:', device)
            net = init_model(ob_size, hidden_size, act_size, model_name).to(device)
            # print(net.lstm.rec_filter)
            net, loss_log, gradient_log, spectral_radius_log, rec_balance_log, lyanpunov_spectrum_log = train(net, epochs, lyapunov_every, lr=lr, wd=wd_ei,
                                                                                                            batch_size=batch_size)
            loss_log_ei[j, seed, :, :] = loss_log
            gradient_log_ei[j, seed, :, :] = gradient_log
            spectral_radius_log_ei[j, seed, :, :] = spectral_radius_log
            rec_balance_log_ei[j, seed, :, :] = rec_balance_log
            lyapunov_spectrum_log_ei[j, seed, :, :] = lyanpunov_spectrum_log

            test_EI_performance[j, seed, :] = test(net)
            net_state_dicts[j][seed] = net.state_dict()

    path = f'./result/multitask/multilr_{model_name}_performance.pkl'
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path))
    info = {'performance': test_EI_performance,
            'loss_log_ei': loss_log_ei,
            'gradient_log_ei': gradient_log_ei,
            'spectral_radius_log_ei': spectral_radius_log_ei,
            'rec_balance_log_ei': rec_balance_log_ei,
            'lyapunov_spectrum_log_ei': lyapunov_spectrum_log_ei,
            'net_state_dicts': net_state_dicts}
    with open(path, 'wb') as f:
        pickle.dump(info, f)
    print(f"Saved performance to {path}")
    return net


In [ ]:
# model_names = ['fixed_eisep', 'entryei']
# model_names = ['eisep']
model_names = ['cstr_simple']
# model_names = ['eisep_init_DBPN']
# lrs = [1e-2]
lrs = [0.5, 1e-1, 5e-2, 1e-2, 5e-3, 1e-3, 5e-4, 1e-4]
for model_name in model_names:
    print(model_name)
    multiple_train_multilr(model_name, lrs)
raise ValueError('stop here')

In [ ]:
from net.common import savefig
def plot_test_performance_all(test_acc, env_names, model_names, colors, legend_names, show_values=True, title=''):
    # test_acc: [n_models, n_seed, n_envs]
    assert test_acc.shape == (len(model_names), n_seed, len(env_names))
    test_acc = np.where(test_acc is None, 0., test_acc)
    mean_acc = np.mean(test_acc, axis=1)
    std_acc = np.std(test_acc, axis=1)/np.sqrt(test_acc.shape[1])

    n_env = mean_acc.shape[1]
    x = np.arange(n_env)
    width = 0.8  # bar width

    fig, ax = plt.subplots(figsize=(20,4))

    for i in range(mean_acc.shape[0]):
        bars = ax.bar(x + (i - mean_acc.shape[0]/2)*width/mean_acc.shape[0], mean_acc[i], width/mean_acc.shape[0],
                      yerr=std_acc[i], capsize=4, color=colors[i],
                      label=legend_names[i])
        if show_values:
            ax.bar_label(bars, fmt='%.2f', padding=2)
    ax.set_xticks(x)
    ax.set_xticklabels(env_names, rotation=30, ha='right')
    ax.set_ylabel('Test accuracy')
    ax.set_title(f'Model comparison across environments. '+title)
    ax.set_xmargin(0.01)
    ax.legend()
    plt.tight_layout()
    plt.savefig('./output/multitask/yang19_test_performance.png', dpi=400)
    plt.show()
    plt.close()

    fig, ax = plt.subplots(figsize=(5,3))
    mean_all = mean_acc.mean(axis=1)
    std_all = std_acc.mean(axis=1)
    for i in range(len(model_names)):
        bars = ax.bar(i, mean_all[i], yerr=std_all[i], capsize=4, color=colors[i])
        if show_values:
            ax.bar_label(bars, fmt='%.2f', padding=2)
    ax.set_xticks(np.arange(len(model_names)))
    ax.set_xticklabels(legend_names[:len(model_names)], rotation=30, ha='right')
    ax.set_ylabel('Test accuracy')
    ax.set_title(f'Model comparison overall. '+title)
    plt.tight_layout()
    plt.savefig('./output/multitask/yang19_test_performance_overall.png', dpi=400)
    plt.show()
    plt.close()

def performance_vs_lr(test_acc, env_names, model_names, colors, legend_names, lrs, show_values=True, title=''):
    # test_acc: [n_models, n_lr, n_seed, n_envs]
    fontsize = 18
    assert test_acc.shape == (len(model_names), len(lrs), n_seed, len(env_names))
    test_acc_flatten = test_acc.reshape(len(model_names), len(lrs), -1)
    mean_acc = np.mean(test_acc_flatten, axis=-1)
    std_acc = np.std(test_acc_flatten, axis=-1)/np.sqrt(test_acc_flatten.shape[-1])
    n_lr = len(lrs)
    # fig, ax = plt.subplots(figsize=(3,3))
    fig, axes = plt.subplots(1, 2, figsize=(8,4))
    ax = axes[0]

    for i in range(len(model_names)):
        ax.plot(lrs, mean_acc[i], marker='o', color=colors[i], label=legend_names[i],
                markeredgecolor='white', markeredgewidth=1, markersize=10)
        ax.fill_between(lrs, mean_acc[i]-std_acc[i], mean_acc[i]+std_acc[i], alpha=0.1, color=colors[i])
    ### replot the first model at the top
    ax.plot(lrs, mean_acc[0], marker='o', color=colors[0],
            markeredgecolor='white', markeredgewidth=1, markersize=10)
    ax.fill_between(lrs, mean_acc[0]-std_acc[0], mean_acc[0]+std_acc[0], alpha=0.1, color=colors[0])
    ax.vlines(1e-2, 0, 1.1, colors='k', linestyles='dashed')
    ax.vlines(5e-3, 0, 1.1, colors='g', linestyles='dashed')
    ax.set_xscale('log')
    ax.set_xlabel('Learning rate', fontsize=fontsize)
    # ax.set_ylabel('Test accuracy', fontsize=fontsize)
    # ax.set_xticklabels(lrs, fontsize=fontsize)
    ax.tick_params(labelsize=fontsize-5)
    ax.set_title('Performance vs. learning rates'+title, fontsize=fontsize)
    # ax.legend(ncol=2, loc='lower left', fontsize=fontsize-5)

    ax = axes[1]
    for i in range(len(model_names)):
        ax.plot(0, 0, marker='o', color=colors[i], label=legend_names[i],
                markeredgecolor='white', markeredgewidth=1, markersize=10)
    ax.set_xlim(1,2)
    ax.legend(fontsize=fontsize-5, loc='center left', framealpha=1, shadow=True, markerscale=1)
    ax.axis('off')

    plt.tight_layout()
    savefig('./output/multitask/yang19_performance_vs_lr.png', dpi=900)
    plt.show()
    plt.close()

    # =========================================================
    # Plot the best performance of each model (best LR selected)
    # =========================================================

    # test_acc: [n_models, n_lr, n_seed, n_envs]
    test_acc_flatten = test_acc.reshape(len(model_names), len(lrs), -1)

    # mean/std over (seed × env)
    mean_acc_lr = np.mean(test_acc_flatten, axis=-1)   # [n_models, n_lr]
    std_acc_lr  = np.std(test_acc_flatten, axis=-1) / np.sqrt(test_acc_flatten.shape[-1])

    # choose best lr for each model
    best_lr_idx = np.argmax(mean_acc_lr, axis=1)       # [n_models]
    best_lrs = [lrs[idx] for idx in best_lr_idx]
    best_mean_acc = mean_acc_lr[np.arange(len(model_names)), best_lr_idx]
    best_std_acc  = std_acc_lr[np.arange(len(model_names)), best_lr_idx]

    fig, ax = plt.subplots(1,1,figsize=(6,3), constrained_layout=True)
    for i in range(len(model_names)):
        bars = ax.bar(i, best_mean_acc[i], yerr=best_std_acc[i], capsize=4, color=colors[i])
        if show_values:
            ax.bar_label(bars, fmt='%.2f', padding=2, fontsize=15)
    ax.set_xticks(np.arange(len(model_names)))
    ax.set_xticklabels(legend_names[:len(model_names)], rotation=30, ha='right', fontsize=12)
    # ax.set_xticklabels(legend_names[:len(model_names)], fontsize=12)
    ax.set_ylabel('Test accuracy', fontsize=15)
    ax.tick_params(labelsize=15, axis='y')
    if title != 'Difference of':
        ax.set_ylim(0,1.2)
    else:
        ax.set_ylim(-0.06,0.06)
    if title == '':
        ax.set_title(f'Multi-task cognitive benchmark performance', fontsize=16)
    else:
        ax.set_title(f'{title} performance averaged across tasks', fontsize=16)
    # plt.tight_layout()
    figname = f'./output/multitask/yang19_test_performance_overall_{title}.png'
    plt.savefig(figname, dpi=400)
    print(f'./output/multitask/yang19_test_performance_overall_{title}.png')
    return

In [ ]:
# model_names = ['eisep', 'simple', 'dann_sgd', 'dann_adam', 'colei_sgd', 'colei_adam', 'dbpn']
model_names = ['eisep', 'simple', 'absolute_eisep', 'fixed_eisep', 'entryei', 'eisep_init_ColEI', 'eisep_init_DBPN', 'cstr_simple']
# model_names = ['eisep', 'simple', 'eisep_relu', 'simple_relu']
# legend_names = ['EISep', 'Simple', 'EntryEI', 'Fixed-EISep', 'Fixed-EntryEI',  'DANN-SGD', 'DANN-Adam', 'ColEI-SGD', 'ColEI-Adam']
# legend_names = ['EISep', 'Simple', 'DANN-SGD', 'DANN-Adam', 'ColEI-SGD', 'ColEI-Adam', 'DBPN']
legend_names = ['EISep', 'Simple', 'Abs-EISep', 'Fixed-EISep', 'EntryEI', 'EISep(N1)', 'EISep(N2)', 'Cstr-Simple']
# legend_names = ['EISep', 'Simple', 'EISep-ReLU', 'Simple-ReLU']
lrs = [0.5, 1e-1, 5e-2, 1e-2, 5e-3, 1e-3, 5e-4, 1e-4]
n_lr = len(lrs)
test_acc = np.zeros((len(model_names), n_lr, n_seed, len(env.envs)))
loss_log = np.zeros((len(model_names), n_lr, n_seed, epochs, 3))
gradient_log = np.zeros((len(model_names), n_lr, n_seed, epochs, 1))
spectral_radius_log = np.zeros((len(model_names), n_lr, n_seed, epochs, 1))
rec_balance_log = np.zeros((len(model_names), n_lr, n_seed, epochs, seq_len))
net_state_dicts = []
# colors = ['C1', 'C0', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9']
colors = ['C1', 'C0', 'C7', 'C8', 'C9', '#E69F00', '#009E73', 'black', ]

for i, model_name in enumerate(model_names):
    print(i)
    path = f'./result/multitask/multilr_{model_name}_performance.pkl'
    with open(path, 'rb') as f:
        info = pickle.load(f)
    test_acc[i] = info['performance']
    loss_log[i] = info['loss_log_ei']
    gradient_log[i] = info['gradient_log_ei']
    spectral_radius_log[i] = info['spectral_radius_log_ei']
    rec_balance_log[i] = info['rec_balance_log_ei']
    net_state_dicts.append(info['net_state_dicts'])
env_names = [e_i.spec.id for e_i in env.envs]
env_names = [x.split('yang19.')[1].split('-v0')[0] for x in env_names]
# print(env_names)
performance_vs_lr(test_acc, env_names, model_names, colors, legend_names, lrs, show_values=True, title='')
# for j, lr in enumerate(lrs):
#     plot_test_performance_all(test_acc[:,j], env_names, model_names, colors, legend_names, show_values=True, title=f'Learning rate {lr}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _find_threshold_intersections_loglr(lrs, y, thr=0.8, eps=1e-12):
    """
    Find intersection points between y(log10(lr)) and horizontal line y=thr
    using piecewise linear interpolation in x=log10(lr).

    Returns:
        lr_left, lr_right, xs (sorted unique log10-intersections)
    Notes:
        - If there are 2+ intersections, we return [leftmost, rightmost].
        - If there is exactly 1 intersection (rare edge case), return a degenerate interval.
        - If no intersection, return (nan, nan).
    """
    lrs = np.asarray(lrs, dtype=float)
    y = np.asarray(y, dtype=float)
    assert lrs.ndim == 1 and y.ndim == 1 and lrs.shape == y.shape

    # sort by lr ascending
    order = np.argsort(lrs)
    lrs = lrs[order]
    y = y[order]
    x = np.log10(lrs)

    xs = []

    for i in range(len(lrs) - 1):
        x0, x1 = x[i], x[i+1]
        y0, y1 = y[i], y[i+1]

        # handle exact threshold hits
        if abs(y0 - thr) < eps:
            xs.append(x0)
            continue
        if abs(y1 - thr) < eps:
            xs.append(x1)
            continue

        s0 = y0 - thr
        s1 = y1 - thr
        if s0 * s1 < 0:  # crossing
            t = (thr - y0) / (y1 - y0)  # in (0,1)
            xs.append(x0 + t * (x1 - x0))

    if len(xs) == 0:
        return np.nan, np.nan, []

    xs = np.array(sorted(xs))
    # de-duplicate
    xs_uniq = [xs[0]]
    for v in xs[1:]:
        if abs(v - xs_uniq[-1]) > 1e-9:
            xs_uniq.append(v)
    xs = xs_uniq

    if len(xs) == 1:
        if y[0] >= thr and y[len(lrs)-1] < thr:
            lr_left = 10 ** x[0]
            lr_right = 10 ** xs[0]
        elif y[0] < thr and y[len(lrs)-1] >= thr:
            lr_left = 10 ** xs[0]
            lr_right = 10 ** x[0]
        else:
            lr_left = lr_right = 10 ** xs[0]
        return float(lr_left), float(lr_right), xs

    lr_left = 10 ** xs[0]
    lr_right = 10 ** xs[-1]
    return float(lr_left), float(lr_right), xs


def plot_lr_performance_and_stability_vertical(
    test_acc, env_names, model_names, colors, legend_names, lrs,
    thr=0.8, robust=True,
    title='', save_path=None,
    lcb_multiplier=1.0,  # 1.0 -> mean - 1*SE; set 1.96 for ~95% LCB if you prefer
):
    """
    Top: performance vs learning rate (mean ± SE).
    Bottom: a single (non-fragmented) stable LR interval per model, computed by
            finding intersections between (mean - k*SE) and y=thr in log10(lr) space.

    test_acc: [n_models, n_lr, n_seed, n_envs]
    """
    fontsize = 15
    n_seed = test_acc.shape[2]
    n_models = len(model_names)
    assert test_acc.shape == (n_models, len(lrs), n_seed, len(env_names))

    # ---- sort learning rates ----
    lrs = np.array(lrs, dtype=float)
    order = np.argsort(lrs)
    lrs = lrs[order]
    test_acc = test_acc[:, order, :, :]

    # ---- aggregate (flatten seeds * envs) ----
    acc_flat = test_acc.reshape(n_models, len(lrs), -1)
    mean_acc = acc_flat.mean(axis=-1)
    se_acc   = acc_flat.std(axis=-1) / np.sqrt(acc_flat.shape[-1])  # standard error

    # choose curve for stability thresholding
    if robust:
        y_stab = mean_acc - lcb_multiplier * se_acc
    else:
        y_stab = mean_acc

    # ---- layout (vertical, shared x) ----
    curve_height = 3.24
    zone_height = 0.25*test_acc.shape[0] + 0.51
    fig, axes = plt.subplots(
        2, 1, figsize=(5, curve_height+zone_height),
        sharex=True,
        gridspec_kw={'height_ratios': [curve_height, zone_height]},
        # constrained_layout=True,
    )
    ax_curve, ax_zone = axes

    # ==================================================
    # Top: performance vs lr
    # ==================================================
    ax_curve.axhline(thr, color='k', linestyle='--', linewidth=1, alpha=0.6)

    for i in range(n_models):
        ax_curve.plot(
            lrs, mean_acc[i], marker='o', color=colors[i],
            markeredgecolor='white', markeredgewidth=1, markersize=9,
            label=legend_names[i]
        )
        ax_curve.fill_between(
            lrs, mean_acc[i] - se_acc[i], mean_acc[i] + se_acc[i],
            alpha=0.12, color=colors[i]
        )

    ax_curve.set_xscale('log')
    ax_curve.set_ylim(0, 1.05)
    ax_curve.set_ylabel('Test performance', fontsize=fontsize)
    ax_curve.tick_params(labelsize=fontsize-5)
    ax_curve.set_title('Performance and stable zone', fontsize=fontsize)

    # ax_curve.legend(
    #     fontsize=fontsize-3, ncol=3, loc='lower left',
    #     framealpha=1.0
    # )

    # ==================================================
    # Bottom: stability zones (single interval per model via intersections)
    # ==================================================
    row_h = 0.8
    y_pos = np.arange(n_models)[::-1]  # top to bottom

    # for placing text on the right in log scale
    text_x = lrs[-1] * 1.35

    intervals = []
    widths_decades = []

    for i, y in enumerate(y_pos):
        lrL, lrR, xs = _find_threshold_intersections_loglr(lrs, y_stab[i], thr=thr)

        # If the curve never crosses the threshold but stays above it everywhere,
        # treat the whole lr range as stable; similarly if always below -> none.
        if not np.isfinite(lrL) or not np.isfinite(lrR):
            if np.all(y_stab[i] >= thr):
                lrL, lrR = float(lrs[0]), float(lrs[-1])
            else:
                lrL, lrR = np.nan, np.nan

        intervals.append((lrL, lrR))

        # draw bar if exists
        if np.isfinite(lrL) and np.isfinite(lrR) and lrR >= lrL:
            ax_zone.broken_barh(
                [(lrL, lrR - lrL)],
                (y - row_h/2, row_h),
                facecolors=colors[i],
                edgecolors=colors[i],
                alpha=0.35,
                linewidth=1.5
            )
            w = np.log10(lrR) - np.log10(lrL) if (lrL > 0 and lrR > 0 and lrR > lrL) else 0.0
        else:
            # faint baseline for "none"
            ax_zone.hlines(y, lrs[0], lrs[-1], color='k', alpha=0.05, linewidth=6)
            w = 0.0

        widths_decades.append(float(w))

        ax_zone.text(
            text_x, y,
            rf"$\Delta_{{\mathrm{{LR}}}}={w:.2f}$",
            va='center', ha='left', fontsize=fontsize-3
        )

    ax_zone.set_xscale('log')
    ax_zone.set_xlim(lrs[0]*0.8, lrs[-1]*1.2)
    ax_zone.set_yticks(y_pos)
    ax_zone.set_yticklabels(legend_names, fontsize=fontsize-2)
    ax_zone.tick_params(axis='x', labelsize=fontsize-3)
    ax_zone.set_xlabel('Learning rate', fontsize=fontsize)

    if robust:
        pass
        # ax_zone.set_title(
        #     # rf"LCB-stable learning-rate interval (intersections in $\log_{{10}}(\mathrm{{lr}})$; "
        #     # rf"$\mathrm{{perf}}-{lcb_multiplier:.2g}\,\mathrm{{SE}}\geq{thr:.1f}$)",
        #     "LCB-stable learning-rate interval",
        #     fontsize=fontsize
        # )
    else:
        ax_zone.set_title(
            rf"Stable learning-rate interval (intersections in $\log_{{10}}(\mathrm{{lr}})$; "
            rf"$\mathrm{{perf}}\geq{thr:.1f}$)",
            fontsize=fontsize
        )

    fig.suptitle(title, fontsize=fontsize+1, y=0.98)
    fig.subplots_adjust(
        left=0.25,
        right=0.8,
        bottom=0.12,
        top=0.92,
        hspace=0.05
    )
    # plt.subplots_adjust(wspace=0.01)
    # plt.tight_layout()

    if save_path is not None:
        savefig(save_path, dpi=400)

    plt.show()
    plt.close()

    # print useful numbers for paper
    for i in range(n_models):
        lrL, lrR = intervals[i]
        print(f"{legend_names[i]}: stable LR ≈ [{lrL:.2e}, {lrR:.2e}], ΔLR={widths_decades[i]:.2f} decades")

    # return values for downstream use
    return mean_acc, se_acc, intervals, widths_decades


# usage
plot_lr_performance_and_stability_vertical(
    test_acc, env_names, model_names, colors, legend_names, lrs,
    thr=0.8, robust=True, lcb_multiplier=1.0,
    title='', save_path='./output/multitask/yang19_performance_vs_lr.pdf'
)


In [ ]:

# net_state_dicts: [model_names, n_lr, n_seed]
hidden_size = 64
n_seed = 5
def plot_sparsity(net_state_dicts, ob_size, hidden_size, act_size, model_names, legend_names, colors, lrs):
    sparsity_ar = np.zeros((len(model_names), len(net_state_dicts[0]), n_seed))
    for i, model_name in enumerate(model_names):
        net = init_model(ob_size, hidden_size, act_size, model_name)
        for j, state_dict in enumerate(net_state_dicts[i]):
            for seed in range(n_seed):
                net.load_state_dict(state_dict[seed])
                weight = net.get_actual_weight().detach().cpu().numpy()
                sparsity = np.mean(np.abs(weight)<0.1*np.std(np.abs(weight)))
                sparsity_ar[i,j,seed] = sparsity
    fig, ax = plt.subplots(figsize=(5,3))
    mean_sparsity = np.mean(sparsity_ar, axis=-1)
    std_sparsity = np.std(sparsity_ar, axis=-1)/np.sqrt(sparsity_ar.shape[2])
    print(mean_sparsity.shape)
    fontsize = 15
    for i in range(len(model_names)):
        # bars = ax.bar(i, mean_sparsity[i], yerr=std_sparsity[i], capsize=4, color=colors[i])
        # ax.bar_label(bars, fmt='%.2f', padding=2)
        ax.plot(lrs, mean_sparsity[i], marker='o', color=colors[i],
                markeredgecolor='white', markeredgewidth=1, markersize=10)
        ax.fill_between(lrs, mean_sparsity[i]-std_sparsity[i], mean_sparsity[i]+std_sparsity[i], alpha=0.1, color=colors[i])
        ax.vlines(1e-2, 0, 1.1, colors='k', linestyles='dashed')
    ax.set_xscale('log')
    ax.set_xlabel('Learning rate', fontsize=fontsize)
    # ax.set_xticks(np.arange(len(model_names)))
    # ax.set_xticklabels(legend_names[:len(model_names)], rotation=30, ha='right')
    ax.set_ylabel('Sparsity')
    ax.set_title(f'Model weight sparsity comparison overall')
    plt.tight_layout()
    plt.savefig('./output/multitask/yang19_multilr_sparsity.png', dpi=400)
    plt.show()
    plt.close()
    return sparsity_ar

In [ ]:
plot_sparsity(net_state_dicts, ob_size, hidden_size, act_size, model_names, legend_names, colors, lrs)

In [ ]:
# path = f'./result/multitask/eisep_performance.pkl'
# with open(path, 'rb') as f:
#     info = pickle.load(f)
# net = Net(ob_size=ob_size, num_h=64, act_size=act_size, entropy_loss=True).to(device)
# net.load_state_dict(info['net'])

# weight = net.get_actual_weight().detach().cpu().numpy()
# sparsity = 1 - np.mean(np.abs(weight)>0.1*np.std(np.abs(weight)))
# eimask = net.lstm.eimask_logit.detach().cpu().numpy()
# # mask = F.gumbel_softmax(net.lstm.eimask_logit, tau=0.1, hard=False, dim=-1).detach().cpu().numpy()
# exc_ratio = np.mean(eimask[:, 0] > eimask[:, 1])
# fig, ax = plt.subplots(figsize=(6,6))
# ax.set_title(f'Excitation ratio: {exc_ratio:.2f}\nSparsity: {sparsity:.2f}')

# im = ax.imshow(weight, cmap='RdBu_r')
# plt.colorbar(im, ax=ax)
# plt.show()

In [ ]:

def plot_loss_curve_all_multilr(n_epoch, colors, model_names, legend_names,
                        loss_log_all, gradient_log_all, spectral_radius_log, rec_balance_log, lrs
                        ):
    # loss_log_all: [n_model, n_lr, n_seed, n_epoch, 3]
    # gradient_log_all: [n_model, n_lr, n_seed, n_epoch, 1]
    # spectral_radius_log: [n_model, n_lr, n_seed, n_epoch, 1]
    # rec_balance_log: [n_model, n_lr, n_seed, n_epoch, seq_len]
    fig, ax = plt.subplots(1,4, figsize=(12,3))
    fontsize = 18
    n_model = len(model_names)
    n_lr = len(lrs)
    ax = ax.flatten()
    mean_loss = np.nanmean(loss_log_all, axis=2).mean(axis=2)
    assert mean_loss.shape == (n_model, n_lr, 3)
    log_mean_loss = np.log(mean_loss + 1e-10)  # Add small constant to avoid log(0)
    std_loss = np.nanstd(loss_log_all, axis=2).mean(axis=2)
    log_std_loss = np.log(std_loss + 1e-10)  # Add small constant to avoid log(0)

    mean_gradient = np.nanmean(gradient_log_all, axis=2).mean(axis=2).squeeze()
    log_mean_gradient = np.log(mean_gradient + 1e-10)  # Add small constant to avoid log(0)
    std_gradient = np.nanstd(gradient_log_all, axis=2).mean(axis=2).squeeze()
    log_std_gradient = np.log(std_gradient + 1e-10)  # Add small constant to avoid log(0)

    mean_spectral_radius = np.nanmean(spectral_radius_log, axis=2).mean(axis=2).squeeze()
    std_spectral_radius = np.nanstd(spectral_radius_log, axis=2).mean(axis=2).squeeze()

    mean_rec_balance = np.nanmean(rec_balance_log, axis=(2,-1)).mean(axis=2)
    log_mean_rec_balance = np.log(mean_rec_balance + 1e-10)  # Add small constant to avoid log(0)
    std_rec_balance = np.nanstd(rec_balance_log, axis=(2,-1)).mean(axis=2)
    log_std_rec_balance = np.log(std_rec_balance + 1e-10)  # Add small constant to avoid log(0)
    print(mean_rec_balance[-1])
    loss_idx = 1
    for i in range(n_model):
        model_name = legend_names[i]
        ax[0].plot(lrs, mean_loss[i, :, loss_idx], label=model_name, color=colors[i], alpha=0.9, marker='o',
                   markeredgecolor='white', markeredgewidth=1, markersize=10)
        ax[0].fill_between(lrs, mean_loss[i, :, loss_idx] - std_loss[i, :, loss_idx], mean_loss[i, :, loss_idx] + std_loss[i, :, loss_idx], alpha=0.1, color=colors[i])
        ax[1].plot(lrs, mean_gradient[i], label=model_name, color=colors[i], alpha=0.9, marker='o',
                   markeredgecolor='white', markeredgewidth=1, markersize=10)
        ax[1].fill_between(lrs, mean_gradient[i] - std_gradient[i], mean_gradient[i] + std_gradient[i], alpha=0.1, color=colors[i])
        ax[2].plot(lrs, mean_spectral_radius[i], label=model_name, color=colors[i], alpha=0.9, marker='o',
                   markeredgecolor='white', markeredgewidth=1, markersize=10)
        ax[2].fill_between(lrs, mean_spectral_radius[i] - std_spectral_radius[i], mean_spectral_radius[i] + std_spectral_radius[i], alpha=0.1, color=colors[i])
        ax[3].plot(lrs, mean_rec_balance[i], label=model_name, color=colors[i], alpha=0.9, marker='o',
                   markeredgecolor='white', markeredgewidth=1, markersize=10)
        ax[3].fill_between(lrs, mean_rec_balance[i] - std_rec_balance[i], mean_rec_balance[i] + std_rec_balance[i], alpha=0.1, color=colors[i])
        # ax[3].fill_between(np.arange(n_epoch), mean_rec_balance[i] - std_rec_balance[i], mean_rec_balance[i] + std_rec_balance[i], alpha=0.1, color=colors[i])
    ###Replot the first model at the top

    ax[0].legend(fontsize=fontsize-5, loc='upper left')
    i = 0
    model_name = legend_names[i]
    ax[0].plot(lrs, mean_loss[i, :, loss_idx], label=model_name, color=colors[i], alpha=0.9, marker='o',
                markeredgecolor='white', markeredgewidth=1, markersize=10)
    ax[0].fill_between(lrs, mean_loss[i, :, loss_idx] - std_loss[i, :, loss_idx], mean_loss[i, :, loss_idx] + std_loss[i, :, loss_idx], alpha=0.1, color=colors[i])
    ax[1].plot(lrs, mean_gradient[i], label=model_name, color=colors[i], alpha=0.9, marker='o',
                markeredgecolor='white', markeredgewidth=1, markersize=10)
    ax[1].fill_between(lrs, mean_gradient[i] - std_gradient[i], mean_gradient[i] + std_gradient[i], alpha=0.1, color=colors[i])
    ax[2].plot(lrs, mean_spectral_radius[i], label=model_name, color=colors[i], alpha=0.9, marker='o',
                markeredgecolor='white', markeredgewidth=1, markersize=10)
    ax[2].fill_between(lrs, mean_spectral_radius[i] - std_spectral_radius[i], mean_spectral_radius[i] + std_spectral_radius[i], alpha=0.1, color=colors[i])
    ax[3].plot(lrs, mean_rec_balance[i], label=model_name, color=colors[i], alpha=0.9, marker='o',
                markeredgecolor='white', markeredgewidth=1, markersize=10)
    ax[3].fill_between(lrs, mean_rec_balance[i] - std_rec_balance[i], mean_rec_balance[i] + std_rec_balance[i], alpha=0.1, color=colors[i])



    ax[0].set_title('Loss', fontsize=fontsize)
    ax[0].set_ylim([1e-2,1e15])
    ax[0].set_yscale('log')
    ax[1].set_title('Gradient norm', fontsize=fontsize)
    ax[1].set_ylim([1e-3,1e15])
    ax[1].set_yscale('log')
    ax[2].set_title('Spectral radius', fontsize=fontsize)
    ax[2].set_ylim([0,30])
    ax[3].set_title('Recurrent balance', fontsize=fontsize)
    ax[3].set_ylim([1e-2,1e13])
    ax[3].set_yscale('log')
    # ax[3].legend(fontsize=fontsize, loc='center left', framealpha=0.5, shadow=True, markerscale=1)
    # ax[3].axis('off')

    # ax[3].set_title('Recurrent balance', fontsize=fontsize)

    for a in ax:
        a.tick_params(labelsize=fontsize-5)
        a.set_xlabel('Learning rate', fontsize=fontsize)
        a.set_xscale('log')



    plt.tight_layout()
    savefig('./output/multitask/yang19_loss_curve.png', dpi=400)
    plt.show()

In [ ]:
plot_loss_curve_all_multilr(epochs, colors, model_names, legend_names, loss_log, gradient_log, spectral_radius_log, rec_balance_log, lrs)

In [ ]:
from cut import signed_spectral_W
def cal_modularity(net_state_dicts, ob_size, hidden_size, act_size, model_names, colors):
    # net_state_dicts: list of list of state_dict
    # model_names: list of model names
    # Returns: Q_all, Frustr_all, Ncut_all: [n_model, n_seed, n_k]
    metric_name = ['Q', 'Frustration', 'Ncut']
    k_min = 2
    k_max = 10
    Q_all = np.zeros((len(model_names), n_seed, k_max - k_min + 1))
    Frustr_all = np.ones((len(model_names), n_seed, k_max - k_min + 1))
    Ncut_all = np.ones((len(model_names), n_seed, k_max - k_min + 1))
    for i, model_name in enumerate(model_names):
        print(model_name)
        net = init(ob_size, hidden_size, act_size, model_name)
        for j, state_dict in enumerate(net_state_dicts[i]):
            net.load_state_dict(state_dict)
            weight = net.get_actual_weight().detach().cpu().numpy()
            print(f'Seed {j}, weight mean: {weight.mean():.4f}, std: {weight.std():.4f}')
            if np.isnan(weight).any():
                print('Weight contains NaN, skip')
                continue
            df, _ = signed_spectral_W(weight, k_min, k_max, restarts=5, seed=0)
            Q_all[i,j] = df['Q'].values
            Frustr_all[i,j] = df['Frustr'].values
            Ncut_all[i,j] = df['Ncut'].values

    return Q_all, Frustr_all, Ncut_all


In [ ]:
Q_all, Frustr_all, Ncut_all = cal_modularity(net_state_dicts, ob_size, hidden_size, act_size, model_names, colors)

In [ ]:
metric_name = ['Q', 'Frustration', 'Ncut']
k_min = 2
k_max = 10
fig, ax = plt.subplots(1,3, figsize=(18,5))
fontsize = 15
ax = ax.flatten()
for i, metric in enumerate([Q_all, Frustr_all, Ncut_all]):
    mean_metric = np.mean(metric, axis=1)
    std_metric = np.std(metric, axis=1)/np.sqrt(metric.shape[1])
    for j in range(len(model_names)):
        ax[i].plot(np.arange(k_min, k_max+1), mean_metric[j], label=model_names[j], color=colors[j], alpha=0.9, marker='o')
        ax[i].errorbar(np.arange(k_min, k_max+1), mean_metric[j], yerr=std_metric[j], color=colors[j], alpha=0.9)
    if i == 0:
        ax[i].set_title('Modularity Q', fontsize=fontsize)
        # ax[i].set_ylim([0, 0.6])
    elif i == 1:
        ax[i].set_title('Frustration', fontsize=fontsize)
        # ax[i].set_ylim([0, 0.5])
    elif i == 2:
        ax[i].set_title('Normalized cut', fontsize=fontsize)
        # ax[i].set_ylim([0, 1.5])
    ax[i].set_xlabel('Number of clusters', fontsize=fontsize)
    ax[i].set_ylabel(metric_name[i], fontsize=fontsize)
    ax[i].tick_params(labelsize=fontsize)
    # ax[i].legend(fontsize=fontsize, loc='lower left')
plt.tight_layout()
plt.savefig('./output/multitask/yang19_modularity.png', dpi=400)
plt.show()
plt.close()

In [ ]:
# from pdm_model import Net, SimpleNet
# device = 'cuda:0'
# batch_size = 64
# hidden_size = 64
# net = Net(ob_size=ob_size, num_h=hidden_size, act_size=act_size, entropy_loss=True).to(device)
# simple_net = SimpleNet(ob_size=ob_size, num_h=hidden_size, act_size=act_size).to(device)
# net_path = './model/multitask/yang19_ei_softplus.pth'
# simple_net_path = './model/multitask/yang19_simple_tanh.pth'
# net.load_state_dict(torch.load(net_path))
# simple_net.load_state_dict(torch.load(simple_net_path))

In [ ]:
n_test = 10
def plot_activity(hidden_size, net, color):
    net.eval()
    fig, ax = plt.subplots((hidden_size-1)//8+1, 8, figsize=(32, (hidden_size//8+1)*3))
    ax = ax.flatten()
    inputs, targets = dataset()
    inputs = torch.from_numpy(inputs).float().to(device)
    targets = torch.from_numpy(targets).long().to(device)

    outputs, eimask_logit, hidden = net(inputs)
    # print('hidden', hidden.shape)
    h_all = hidden[0].detach().cpu().numpy()
    h_all_mean = h_all.mean(1)
    h_all_std = h_all.std(1)
    for j in range(hidden_size):
        ax[j].plot(h_all_mean[:,j], color=color)
        ax[j].fill_between(np.arange(seq_len), h_all_mean[:,j]-h_all_std[:,j], h_all_mean[:,j]+h_all_std[:,j],
                           alpha=0.2, color=color)
    normalized_variance = (h_all_std/(h_all.max((0,1))-h_all.min((0,1)))).mean()
    # normalized_variance = h_all_std.mean()

    plt.suptitle(f'Normalized variance: {normalized_variance:.4f}', fontsize=30)
    plt.tight_layout()
    plt.show()
    plt.close()


In [ ]:
plot_activity(hidden_size, net, 'C0')

In [ ]:
plot_activity(hidden_size, simple_net, 'C1')